# Invoice processing allocation
Self-contained optimization example; no invoice workbook is required. This model's 5,000-unit equality counts processing operations **including rework**, not 5,000 distinct invoices. That is the original coursework assumption. It would need a different equality if 5,000 meant unique invoices. Costs and capacities are supplied assumptions, not estimates from the missing dataset.




We allocate **5,000 invoices** to 3 branches:
- **Amman (A)** cost = 0.90 JD per invoice  
- **Irbid (I)** cost = 0.60 JD per invoice  
- **Aqaba (Q)** cost = 0.30 JD per invoice  
- Aqaba requires **rework** for **10%** of its invoices, and each rework costs **0.30 JD**

So Aqaba total cost per invoice = 0.30 + 0.30×0.10 = **0.33 JD**.

**Constraints**
- Aqaba capacity: **Q ≤ 2000**
- Amman capacity: **A ≤ 3000**
- Accuracy requirement: **A + I ≥ 2000** (40% of 5000)
- Total processed including rework: **A + I + Q + 0.10Q = 5000** → **A + I + 1.10Q = 5000**
- A, I, Q ≥ 0


## Part 1 — Linear Programming (SciPy linprog)

In [1]:
from scipy.optimize import linprog
import numpy as np

# Objective: minimize cost Z = 0.90*A + 0.60*I + 0.33*Q
c = [0.90, 0.60, 0.33]

# Equality: A + I + 1.10Q = 5000
A_eq = [[1, 1, 1.10]]
b_eq = [5000]

# Inequalities (A_ub x <= b_ub):
# A <= 3000
# Q <= 2000
# A + I >= 2000  ->  -A - I <= -2000
A_ub = [
    [1, 0, 0],
    [0, 0, 1],
    [-1, -1, 0]
]
b_ub = [3000, 2000, -2000]

bounds = [(0, 3000), (0, None), (0, 2000)]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

assert result.success, result.message
print("LP Solution:")
print(f" - Amman (A): {result.x[0]:.2f} invoices")
print(f" - Irbid (I): {result.x[1]:.2f} invoices")
print(f" - Aqaba (Q): {result.x[2]:.2f} invoices")

rework = 0.10 * result.x[2]
total_cost = 0.90*result.x[0] + 0.60*result.x[1] + 0.30*result.x[2] + 0.30*rework

print(f" - Rework (0.10Q): {rework:.2f} invoices")
print(f"Minimum Total Cost: {total_cost:.2f} JD")


LP Solution:
 - Amman (A): 0.00 invoices
 - Irbid (I): 2800.00 invoices
 - Aqaba (Q): 2000.00 invoices
 - Rework (0.10Q): 200.00 invoices
Minimum Total Cost: 2340.00 JD


## Part 2 — Genetic Algorithm (Template: Lab 12 GA)

In [2]:
import pygad
import numpy as np
import matplotlib.pyplot as plt

# Cost function (same as LP)
def total_cost(A, I, Q):
    R = 0.10 * Q
    return 0.90*A + 0.60*I + 0.30*Q + 0.30*R  # = 0.90A + 0.60I + 0.33Q

# We encode solutions as (A, Q) only.
# Then compute I from the equality: A + I + 1.10Q = 5000  ->  I = 5000 - A - 1.10Q
# To keep I an integer, Q must be a multiple of 10 (because 1.10Q = 11Q/10).
def fitness_func(ga_instance, solution, solution_idx):
    A, Q = solution

    # Force integer values (lab style)
    A = int(round(A))
    Q = int(round(Q))

    # Enforce Q multiple of 10 (so I becomes integer)
    Q = int(round(Q / 10.0) * 10)

    # Compute I from equality
    I = 5000 - A - int(round(1.10 * Q))  # 1.10Q is integer now

    # Bounds / non-negativity
    if A < 0 or I < 0 or Q < 0:
        return -999999
    if A > 3000 or Q > 2000:
        return -999999

    # Accuracy requirement
    if (A + I) < 2000:
        return -999999

    # Double-check equality (integer-safe)
    if (A + I + int(round(1.10 * Q))) != 5000:
        return -999999

    cost = total_cost(A, I, Q)
    return -cost  # minimize cost -> maximize -cost

# GA instance (Lab 12 style)
ga_instance = pygad.GA(
    num_generations=250,
    sol_per_pop=50,
    num_parents_mating=20,
    fitness_func=fitness_func,
    num_genes=2,           # [A, Q]
    gene_type=int,
    gene_space=[
        {'low': 0, 'high': 3001},      # A
        {'low': 0, 'high': 2001, 'step': 10}  # Q must be multiple of 10
    ],
    mutation_num_genes=1,
    random_seed=42,
    keep_elitism=2
)

# Run GA
ga_instance.run()

# Decode best solution
solution, solution_fitness, solution_idx = ga_instance.best_solution()

A = int(round(solution[0]))
Q = int(round(solution[1] / 10.0) * 10)
I = 5000 - A - int(round(1.10 * Q))
R = int(round(0.10 * Q))

print("GA Best Solution:")
print(f" - Amman (A): {A} invoices")
print(f" - Irbid (I): {I} invoices")
print(f" - Aqaba (Q): {Q} invoices")
print(f" - Rework (0.10Q): {R} invoices")
print(f"Total Cost (JD): {-solution_fitness:.2f}")


GA Best Solution:
 - Amman (A): 3 invoices
 - Irbid (I): 2797 invoices
 - Aqaba (Q): 2000 invoices
 - Rework (0.10Q): 200 invoices
Total Cost (JD): 2340.90


In [3]:
# GA Convergence Curve (same idea as Lab 12)
best_fitness_values = ga_instance.best_solutions_fitness
best_cost_values = [-f for f in best_fitness_values]

# Remove penalty values (very large numbers) for a readable curve
best_cost_values_clean = [c for c in best_cost_values if c < 100000]  # costs should be far below this

plt.figure(figsize=(8,5))
plt.plot(best_cost_values_clean, marker='o')
plt.title("GA Convergence Curve")
plt.xlabel("Generation")
plt.ylabel("Total Cost (JD)")
plt.grid(True)
plt.show()


<Figure size 800x500 with 1 Axes>

In [4]:
from pathlib import Path
import json
assert A + I + 1.10 * Q == 5000
assert A + I >= 2000 and 0 <= A <= 3000 and 0 <= Q <= 2000 and I >= 0
summary = {'lp_minimum_cost_jod': float(result.fun), 'ga_cost_jod': float(-solution_fitness),
           'ga_gap_jod': float(-solution_fitness - result.fun),
           'ga_allocation': {'Amman': A, 'Irbid': I, 'Aqaba': Q},
           'workload_includes_rework': True}
Path('results').mkdir(exist_ok=True)
Path('results/prescriptive_results.json').write_text(json.dumps(summary, indent=2))
print(summary)


{'lp_minimum_cost_jod': 2340.0, 'ga_cost_jod': 2340.9, 'ga_gap_jod': 0.900000000000091, 'ga_allocation': {'Amman': 3, 'Irbid': 2797, 'Aqaba': 2000}, 'workload_includes_rework': True}
